In [1]:

import os
import sys

# Configure Python worker before starting Spark
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("RDD Lab Assignments") \
    .master("local[1]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.pyspark.python", sys.executable) \
    .config("spark.python.worker.reuse", "false") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

sc = spark.sparkContext

print("Spark version:", spark.version)
print("Python executable:", sys.executable)
print("Spark started successfully!")


Spark version: 4.2.0
Python executable: C:\Users\samik\anaconda3\python.exe
Spark started successfully!


In [2]:
import sys
import pyspark

print("Python version:", sys.version)
print("PySpark version:", pyspark.__version__)

Python version: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]
PySpark version: 4.2.0


In [3]:
import os
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot


In [4]:

import os
from pyspark.sql import SparkSession

# Set local hostname resolution for Spark
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

spark = SparkSession.builder \
    .appName("RDD Lab Assignments") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

sc = spark.sparkContext

print("Spark version:", spark.version)
print("Spark started successfully!")


Spark version: 4.2.0
Spark started successfully!


In [9]:
#1.Create RDDs in three different ways
rdd1 = sc.parallelize([10, 20, 30, 40, 50])
print(rdd1.collect())


rdd2 = sc.textFile("input.txt")
print(rdd2.collect())


rdd3 = rdd1.map(lambda x: x * 2)
print(rdd3.collect())

[10, 20, 30, 40, 50]
['Hello Spark', 'Hello Python', 'Big Data Analytics', 'Spark is powerful']
[20, 40, 60, 80, 100]


In [10]:
#2.Read a text file and count the words
rdd = sc.textFile("input.txt")

word_count = rdd.flatMap(lambda line: line.split()) \
                .count()

print("Total words:", word_count)

Total words: 10


In [11]:
#3.Find the frequency of each word in a file
rdd = sc.textFile("input.txt")

word_frequency = rdd.flatMap(lambda line: line.split()) \
                    .map(lambda word: (word.lower(), 1)) \
                    .reduceByKey(lambda a, b: a + b)

print(word_frequency.collect())

[('hello', 2), ('spark', 2), ('python', 1), ('big', 1), ('data', 1), ('analytics', 1), ('is', 1), ('powerful', 1)]


In [12]:
#4.Convert all words to uppercase
rdd = sc.textFile("input.txt")

result = rdd.map(lambda line: line.upper())

result.collect()

['HELLO SPARK', 'HELLO PYTHON', 'BIG DATA ANALYTICS', 'SPARK IS POWERFUL']

In [13]:
#5.Convert all words to lowercase

rdd = sc.textFile("input.txt")

result = rdd.map(lambda line: line.lower())

result.collect()

['hello spark', 'hello python', 'big data analytics', 'spark is powerful']

In [14]:
#6.Capitalize the first letter of each word

rdd = sc.textFile("input.txt")

result = rdd.map(
    lambda line: " ".join(word.capitalize() for word in line.split())
)

result.collect()

['Hello Spark', 'Hello Python', 'Big Data Analytics', 'Spark Is Powerful']

In [15]:
#7. Count occurrences of a given word

rdd = sc.textFile("input.txt")

word = "spark"

count = rdd.flatMap(lambda line: line.split()) \
           .filter(lambda w: w.lower() == word.lower()) \
           .count()

print("Occurrences:", count)

Occurrences: 2


In [17]:
#8.Select sentences containing a given word
rdd = sc.textFile("input.txt")

word = "spark"

result = rdd.filter(
    lambda sentence: word.lower() in sentence.lower().split()
)

print(result.collect())



['Hello Spark', 'Spark is powerful']


In [18]:
#9.Find the longest word
words = ["apple", "elephant", "banana", "computer", "cat"]

rdd = sc.parallelize(words)

longest = rdd.reduce(
    lambda a, b: a if len(a) >= len(b) else b
)

print("Longest word:", longest)
print("Length:", len(longest))

Longest word: elephant
Length: 8


In [19]:
#10.Map registration numbers to corresponding branches

reg_numbers = [58001, 57002, 38003, 39004, 47005]

rdd = sc.parallelize(reg_numbers)

def get_branch(reg):
    series = reg // 1000

    branches = {
        58: "BDA",
        57: "AIML",
        38: "VLSI",
        39: "ES",
        47: "CDC"
    }

    return (reg, branches.get(series, "Unknown"))

result = rdd.map(get_branch)

print(result.collect())

[(58001, 'BDA'), (57002, 'AIML'), (38003, 'VLSI'), (39004, 'ES'), (47005, 'CDC')]


In [29]:
#11.Find maximum, minimum, sum and mean of numbers in a file
rdd = sc.textFile("numbers.txt")

numbers = rdd.flatMap(lambda line: line.split()) \
             .map(int)

maximum = numbers.max()
minimum = numbers.min()
total = numbers.sum()
count = numbers.count()
mean = total / count

print("Maximum:", maximum)
print("Minimum:", minimum)
print("Sum:", total)
print("Mean:", mean)

Maximum: 50
Minimum: 5
Sum: 170
Mean: 24.285714285714285


In [28]:
#12. Compress citizen data by replacing state names with state codes
citizens = sc.textFile("citizen.txt")

state_rdd = sc.textFile("state_codes.txt") \
    .map(lambda line: line.split(",")) \
    .map(lambda x: (x[0], x[1]))

citizen_rdd = citizens.map(lambda line: line.split(","))


# Convert citizens into (state, citizen_record)
citizen_by_state = citizen_rdd.map(
    lambda x: (x[4], x)
)

# Replace full state name with state code
result = citizen_by_state.join(state_rdd).map(
    lambda x: ",".join(x[1][0][:4] + [x[1][1]])
)

print(result.collect())

['Priya,02-02-2001,9876543211,priya@gmail.com,KL', 'Rahul,01-01-2000,9876543210,rahul@gmail.com,KA', 'Amit,03-03-2002,9876543212,amit@gmail.com,TN']
